# Oryza Vision — Training Hama Padi (Khusus Colab)

**Satu notebook, satu model khusus hama.** Model penyakit dari `Oryza_Vision_Colab_Auto_HuggingFace_v3.ipynb` tetap terpisah dan tidak diubah.

**Dataset:** [Project-AgML/CT_PestNet_crop_pest_classification](https://huggingface.co/datasets/Project-AgML/CT_PestNet_crop_pest_classification) di Hugging Face. **Target tiga kelas:** Brown Planthopper (wereng cokelat), Rice Leafhopper (wereng daun padi), dan Rice Leaf Roller (penggulung daun). Ini **identifikasi hama dari foto hama**, bukan diagnosis gejala kerusakan tanaman.

**Urutan:** Runtime > Change runtime type > T4 GPU (jika tersedia) > Run all. Dataset diunduh otomatis melalui streaming Hugging Face. Hasil backup ke `MyDrive/Oryza-Vision/pest-training/`.

> Penting: pembagian data dalam notebook berdasarkan gambar (tidak ada ID tanaman/sesi yang pasti) sehingga metrik test adalah **hasil eksperimen awal**, bukan jaminan performa lapangan. Periksa kemiripan/label manual sebelum deployment.

## 1. Instal dependensi

In [ ]:
%pip -q install -U "datasets>=3.0" huggingface_hub pillow scikit-learn pandas matplotlib tqdm

## 2. Konfigurasi dan koneksi Drive

In [ ]:
import os, io, json, hashlib, random, shutil, collections, time
from pathlib import Path
import numpy as np, pandas as pd
from PIL import Image, ImageOps, UnidentifiedImageError
import torch, torchvision
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets as tvdatasets, transforms, models
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE, '| Torch:', torch.__version__)

# Satu model hama; pilihan architecture berikut bisa diganti
ARCH = 'efficientnet_b0'  # 'efficientnet_b0' atau 'mobilenet_v3_large'
EPOCHS = 10
BATCH_SIZE = 32
LEARNING_RATE = 1e-4
IMAGE_SIZE = 224
MIN_SIDE = 96
MAX_PER_CLASS = 400  # maksimal 400 gambar hama per kelas
TRAIN_RATIO, VAL_RATIO, TEST_RATIO = .70, .15, .15
USE_DRIVE = True
LOCAL = Path('/content/oryza_vision_pest')
RAW = LOCAL / 'curated'; SPLIT = LOCAL / 'splits'; OUT = LOCAL / 'output'
for d in [RAW,SPLIT,OUT]: d.mkdir(parents=True, exist_ok=True)
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_OUT = Path('/content/drive/MyDrive/Oryza-Vision/pest-training')
    DRIVE_OUT.mkdir(parents=True, exist_ok=True)
else: DRIVE_OUT = None

## 3. Download otomatis dan kurasi 3 kelas hama

Mengambil gambar langsung dari Hugging Face (tanpa download manual), memvalidasi ukuran, membuang gambar rusak dan duplikat piksel identik. Label dipetakan dari metadata `ClassLabel` dataset.

**Catatan:** sampling dibatasi 400 per kelas untuk eksperimen yang seimbang; ini tidak menjamin bebas kemiripan visual atau label salah.

In [ ]:
REPO = 'Project-AgML/CT_PestNet_crop_pest_classification'
MAPPING = {
    'brown plant hopper': 'Brown_Planthopper',
    'rice leafhopper': 'Rice_Leafhopper',
    'rice leaf roller': 'Rice_Leaf_Roller'
}

def label_name(row, feature):
    val=row['label']
    if isinstance(val, str): return val
    if hasattr(feature, 'int2str'): return feature.int2str(int(val))
    raise RuntimeError(f'Unsupported label: {val}')

def to_pil(value):
    if isinstance(value, Image.Image): return value
    if isinstance(value, dict):
        if value.get('bytes'): return Image.open(io.BytesIO(value['bytes']))
        if value.get('path'): return Image.open(value['path'])
    raise ValueError('Format image tidak dikenal')

def fetch_dataset():
    repo, mapping=REPO, MAPPING
    root=RAW/'pest'
    counts=collections.Counter()
    drop=collections.Counter()
    hashes=set()
    for dirname in mapping.values():
        (root/dirname).mkdir(parents=True,exist_ok=True)
        existing=list((root/dirname).glob('*.jpg'))
        # agar rerun tetap konsisten, kurasi dibuat ulang
        for p in existing: p.unlink()
    ds=load_dataset(repo, split='train', streaming=True)
    print('Contoh kelas sumber:', [label_feature for label_feature in ds.features['label'].names[:18]])
    ds=ds.shuffle(seed=SEED, buffer_size=3000)
    label_feature=ds.features['label']
    target_total=len(mapping)*MAX_PER_CLASS
    progress=tqdm(total=target_total,desc='Unduh hama')
    try:
        for row in ds:
            name=label_name(row,label_feature)
            if name not in mapping: continue
            target=mapping[name]
            if counts[target]>=MAX_PER_CLASS: continue
            try:
                im=ImageOps.exif_transpose(to_pil(row['image'])).convert('RGB')
                if min(im.size)<MIN_SIDE:
                    drop['too_small']+=1; continue
                # hash piksel agar metadata / format berbeda dengan piksel identik tetap terdeteksi
                sample=im.resize((224,224))
                h=hashlib.sha256(sample.tobytes()).hexdigest()
                if h in hashes:
                    drop['duplicate']+=1; continue
                hashes.add(h)
                filename=root/target/f'{counts[target]:05d}_{h[:12]}.jpg'
                im.save(filename,format='JPEG',quality=90)
                counts[target]+=1;progress.update(1)
            except Exception as err:
                drop['unreadable']+=1
            if sum(counts.values())>=target_total: break
    finally: progress.close()
    print('Hama:', dict(counts), 'dibuang:',dict(drop))
    if any(counts[x]<50 for x in mapping.values()):
        raise RuntimeError('Ada kelas kurang dari 50 gambar. Periksa akses dataset, nama kelas, dan sampel sebelum training.')
    (root/'audit.json').write_text(json.dumps({'counts':dict(counts),'rejected':dict(drop),'repository':repo},indent=2))
    return counts

pest_counts=fetch_dataset()

## 4. Pembagian data 70:15:15

Split per kelas dan simpan manifest; file yang benar-benar identik sudah dihapus sebelum pembagian. Dataset asli memiliki metadata split, tetapi subset hasil sampling ini menggunakan split acak baru sehingga butuh evaluasi lapangan terpisah.

In [ ]:
def split_dataset(task):
    src=RAW/task; dst=SPLIT/task
    if dst.exists(): shutil.rmtree(dst)
    manifest=[]
    for clsdir in sorted(x for x in src.iterdir() if x.is_dir()):
        files=sorted(clsdir.glob('*.jpg'))
        if len(files)<10: raise ValueError(f'Terlalu sedikit data untuk {clsdir.name}')
        train,hold=train_test_split(files,test_size=VAL_RATIO+TEST_RATIO,random_state=SEED)
        val,test=train_test_split(hold,test_size=TEST_RATIO/(VAL_RATIO+TEST_RATIO),random_state=SEED)
        for phase, subset in [('train',train),('val',val),('test',test)]:
            directory=dst/phase/clsdir.name;directory.mkdir(parents=True,exist_ok=True)
            for p in subset:
                shutil.copy2(p,directory/p.name)
                manifest.append({'task':task,'split':phase,'class':clsdir.name,'file':p.name})
    pd.DataFrame(manifest).to_csv(OUT/f'{task}_manifest.csv',index=False)
    print(task,'\n',pd.crosstab(pd.DataFrame(manifest)['class'],pd.DataFrame(manifest)['split']))
split_dataset('pest')

## 5. Training EfficientNet-B0 (atau MobileNetV3-Large) dan evaluasi test

Pemilihan checkpoint berdasarkan *validation macro F1*. Evaluasi test dilakukan setelah training memakai checkpoint terbaik.

In [ ]:
train_tf=transforms.Compose([
    transforms.RandomResizedCrop(IMAGE_SIZE,scale=(.8,1.0)),
    transforms.RandomHorizontalFlip(),transforms.RandomRotation(15),
    transforms.ColorJitter(.12,.12,.12,.04),
    transforms.ToTensor(),transforms.Normalize([.485,.456,.406],[.229,.224,.225])])
eval_tf=transforms.Compose([
    transforms.Resize(256),transforms.CenterCrop(IMAGE_SIZE),
    transforms.ToTensor(),transforms.Normalize([.485,.456,.406],[.229,.224,.225])])

def make_model(arch,n):
    if arch=='efficientnet_b0':
        m=models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
        m.classifier[1]=nn.Linear(m.classifier[1].in_features,n)
    elif arch=='mobilenet_v3_large':
        m=models.mobilenet_v3_large(weights=models.MobileNet_V3_Large_Weights.DEFAULT)
        m.classifier[3]=nn.Linear(m.classifier[3].in_features,n)
    else:raise ValueError('ARCH tidak dikenali')
    return m.to(DEVICE)

def evaluate(model,loader):
    model.eval(); truths=[];preds=[]
    with torch.no_grad():
        for x,y in loader:
            z=model(x.to(DEVICE));preds.extend(z.argmax(1).cpu().tolist());truths.extend(y.tolist())
    return np.array(truths),np.array(preds)

def train_task(task):
    path=SPLIT/task
    data={phase:tvdatasets.ImageFolder(path/phase,transform=train_tf if phase=='train' else eval_tf) for phase in ('train','val','test')}
    classes=data['train'].classes
    assert all(data[x].classes==classes for x in data)
    loaders={phase:DataLoader(data[phase],batch_size=BATCH_SIZE,shuffle=(phase=='train'),num_workers=2,pin_memory=DEVICE=='cuda') for phase in data}
    model=make_model(ARCH,len(classes))
    # finetune seluruh model
    optimizer=torch.optim.AdamW(model.parameters(),lr=LEARNING_RATE,weight_decay=1e-4)
    loss_fn=nn.CrossEntropyLoss()
    best=-1; history=[]
    checkpoint=OUT/f'{task}_{ARCH}_best.pth'
    for epoch in range(1,EPOCHS+1):
        model.train();running=0
        for images,labels in tqdm(loaders['train'],desc=f'{task} epoch {epoch}/{EPOCHS}',leave=False):
            images,labels=images.to(DEVICE),labels.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            out=model(images);loss=loss_fn(out,labels)
            loss.backward();optimizer.step();running+=loss.item()*len(labels)
        truth,pred=evaluate(model,loaders['val'])
        score=f1_score(truth,pred,average='macro',zero_division=0)
        record={'epoch':epoch,'train_loss':running/len(data['train']),'val_accuracy':accuracy_score(truth,pred),'val_macro_f1':score}
        history.append(record);print(task,record)
        if score>best:
            best=score
            torch.save({'model_state':model.state_dict(),'classes':classes,'architecture':ARCH,'image_size':IMAGE_SIZE,'task':task,'best_val_f1':best},checkpoint)
    pd.DataFrame(history).to_csv(OUT/f'{task}_{ARCH}_history.csv',index=False)
    state=torch.load(checkpoint,map_location=DEVICE,weights_only=True)
    model.load_state_dict(state['model_state'])
    truth,pred=evaluate(model,loaders['test'])
    report=classification_report(truth,pred,labels=list(range(len(classes))),target_names=classes,output_dict=True,zero_division=0)
    (OUT/f'{task}_{ARCH}_test_metrics.json').write_text(json.dumps(report,indent=2))
    cm=confusion_matrix(truth,pred,labels=list(range(len(classes))))
    pd.DataFrame(cm,index=classes,columns=classes).to_csv(OUT/f'{task}_{ARCH}_confusion_matrix.csv')
    print('\nTEST',task,'accuracy',round(accuracy_score(truth,pred),4),'macro F1',round(f1_score(truth,pred,average='macro'),4))
    print(classification_report(truth,pred,labels=list(range(len(classes))),target_names=classes,zero_division=0))
    del model
    if torch.cuda.is_available():torch.cuda.empty_cache()

train_task('pest')

## 6. Simpan hasil ke Google Drive

In [ ]:
if DRIVE_OUT:
    for path in OUT.iterdir():
        if path.is_file(): shutil.copy2(path, DRIVE_OUT/path.name)
    print('Hasil tersimpan di:',DRIVE_OUT)
else: print('Hasil ada di',OUT)
print('File:',[x.name for x in OUT.iterdir()])

## 7. Artefak dan penggunaan

File utama: `pest_efficientnet_b0_best.pth` (atau sesuai `ARCH`), `pest_manifest.csv`, `pest_*_history.csv`, `pest_*_test_metrics.json`, dan `pest_*_confusion_matrix.csv`. File tersimpan di `MyDrive/Oryza-Vision/pest-training/`.

Checkpoint PyTorch `.pth` **belum dapat langsung dipakai di Flutter/TFLite**; diperlukan ekspor/konversi tersendiri. Model penyakit tetap menggunakan notebook dan checkpoint lama.

**Batasan:** foto dengan beberapa spesies atau hama terlalu kecil tidak ideal untuk classifier satu-label. Lakukan inspeksi manual terhadap gambar dan uji foto hasil kamera asli sebelum digunakan.